# m06_l02_breaker

Exobean Engineering. Run the cells from top to bottom.

In [ ]:
# Google Colab only: fetches the free code companion, then run the rest.
import io, os, sys, urllib.request, zipfile
URL = "https://github.com/exobean/ai-agents-level3/releases/latest/download/prodlab-companion.zip"
if "google.colab" in sys.modules and not os.path.isdir("shopassist"):
    try:
        data = io.BytesIO(urllib.request.urlopen(URL).read())
    except OSError:  # network blocks the download: upload the zip
        from google.colab import files
        data = next(iter(files.upload()))
    zipfile.ZipFile(data).extractall(".")
    os.chdir(next(d for d in os.listdir(".") if d.endswith("-companion")))
    sys.path.insert(0, os.getcwd())


## Breaker

In [ ]:
from prodlab.perf import FakeClock
from prodlab.reliability import CircuitBreaker, FlakyProvider, serve

clock = FakeClock()
provider = FlakyProvider(clock, seed=2, fail_rate=0.0, slow_rate=0.0,
                         outage=(30, 70))     # down from 30 s to 70 s
breaker = CircuitBreaker(clock, threshold=3, cool_down_s=20)
rows = serve(provider, breaker, clock, n=60, every_s=2.0)
last = None
for r in rows:
    if r["state"] != last:                    # print state changes
        t, state = r["t"], r["state"]
        print(f"t={t:5.1f}s  breaker {state:<9} {r['outcome']}")
        last = r["state"]
print("provider calls:", provider.calls, "of", len(rows), "requests")
